# 第 1 课｜从下一个 Token 到会调用工具的 Agent

**核心问题：** 模型输出了工具名和参数，现实中的函数究竟有没有运行？

这份 Notebook 不调用在线模型，不需要账号或密钥。我们使用固定输入和可计数工具，把同一条运行轨迹逐层展开。完成后，你应能指出：谁生成请求、谁检查、哪一行产生副作用、结果怎样进入下一轮、程序为何停止。


## 运行约定与路线

- 从空 kernel 按顺序运行；不要跳过断言。
- 代码标识符和事件字段使用英文，教学解释使用中文。
- 所有时间、模型输出和失败路径都固定，因此同学之间可以逐项比较。
- 核心实验顺序：`token → proposal → registry/schema → policy → tool → observation → stop → MCP`。

本课前期属于手写阶段。学生版会把 Registry/Policy 和 Runtime 核心答案替换为 TODO，但保留数据模型、工具、模型 stub 与公开测试。


## 预测 0｜先不要运行工具

下面的模拟模型会返回：

```json
{"kind":"tool_call","name":"get_local_time","arguments":{"zone":"Asia/Shanghai"}}
```

请先写下你的预测：输出出现后，`FixedClock.calls` 应为 `0` 还是 `1`？哪一行代码才可能改变它？


In [ ]:
from __future__ import annotations

import json
import math
from collections.abc import Callable
from dataclasses import dataclass
from datetime import datetime
from enum import StrEnum
from typing import Any, Literal, Protocol

from pydantic import BaseModel, Field, ValidationError


class ToolCall(BaseModel):
    kind: Literal["tool_call"] = "tool_call"
    call_id: str
    name: str
    arguments: dict[str, Any] = Field(default_factory=dict)


class FinalAnswer(BaseModel):
    kind: Literal["final"] = "final"
    answer: str


Proposal = ToolCall | FinalAnswer


class ToolObservation(BaseModel):
    call_id: str
    tool_name: str
    ok: bool
    value: Any | None = None
    error: str | None = None


class Model(Protocol):
    def respond(self, messages: list[dict[str, Any]]) -> Proposal: ...


## Python 代码地图｜先看对象，再看 Agent

这一格专门把 Python 基础放回本课数据流。变量是名字与对象的绑定，函数接收参数并返回结果，`dataclass` 表示固定字段的内部记录，Pydantic Model 在边界处校验数据；类可以保存状态和方法，继承可以复用接口，但本课不会为了展示继承而堆叠类层次。

下面的 `LLMRequest`/`LLMResponse` 是课程内部的中立接口，不连接在线模型。它们帮助我们把不同供应商的请求格式统一到同一条 Runtime 数据流：工具声明进入请求，模型返回 proposal，应用执行后再把 observation 放回下一次请求。


In [ ]:
@dataclass(frozen=True)
class TraceLabel:
    stage: str
    payload_type: str


def make_trace_label(stage: str, *, payload_type: str) -> TraceLabel:
    return TraceLabel(stage=stage, payload_type=payload_type)


class Capability:
    def describe(self) -> str:
        return "generic capability"


class ReadOnlyCapability(Capability):
    def describe(self) -> str:
        return "read-only capability"


class ToolDeclaration(BaseModel):
    name: str
    description: str
    parameters: dict[str, Any]


class LLMRequest(BaseModel):
    messages: list[dict[str, Any]]
    tools: list[ToolDeclaration] = Field(default_factory=list)


class LLMResponse(BaseModel):
    proposal: Proposal
    stop_reason: Literal["tool_use", "stop"]


class ProviderAdapter(Protocol):
    def complete(self, request: LLMRequest) -> LLMResponse: ...


label = make_trace_label("model.proposed", payload_type=ToolCall.__name__)
request = LLMRequest(
    messages=[{"role": "user", "content": "上海现在几点？"}],
    tools=[
        ToolDeclaration(
            name="get_local_time",
            description="Read the local course clock.",
            parameters={
                "type": "object",
                "properties": {"zone": {"type": "string"}},
                "required": ["zone"],
            },
        )
    ],
)
response = LLMResponse(
    proposal=ToolCall(
        call_id="wire-1",
        name="get_local_time",
        arguments={"zone": "Asia/Shanghai"},
    ),
    stop_reason="tool_use",
)

print({"label": label, "request_tools": [tool.name for tool in request.tools]})
print({"response": response.model_dump(mode="json")})
print({"inheritance_example": ReadOnlyCapability().describe()})
assert response.stop_reason == "tool_use"
assert response.proposal.name == "get_local_time"


## 实验 1｜Token 与下一步概率只解释“模型输出什么”

真实 tokenizer 和 Transformer 很复杂；这里用教学数据观察两件事：切分会改变序列长度，解码器会根据已有 token 选择下一 token。这个实验不代表任何具体商用模型。


In [ ]:
text = "调用时间工具"
character_tokens = list(text)
toy_subword_tokens = ["调用", "时间", "工具"]

next_token_probabilities = {"5": 0.72, "6": 0.11, "4": 0.08, "<eos>": 0.09}
selected_token = max(next_token_probabilities, key=next_token_probabilities.get)

print({"characters": character_tokens, "count": len(character_tokens)})
print({"toy_subwords": toy_subword_tokens, "count": len(toy_subword_tokens)})
print({"context": "2+3=", "selected_token": selected_token})

assert selected_token == "5"
assert len(character_tokens) != len(toy_subword_tokens)


`5` 是一次 token 选择的结果。它可能正确，但这个输出不能证明 Python 加法函数运行过。接着用一个二维注意力示意观察“相关工具更影响当前表示”，并马上检查它没有提供权限。


In [ ]:
query = (1.0, 0.0)  # 教学含义：“现在几点”
keys = {
    "get_local_time": (0.95, 0.05),
    "minutes_until": (0.55, 0.45),
    "delete_calendar": (0.10, 0.90),
}


def dot(left: tuple[float, float], right: tuple[float, float]) -> float:
    return sum(a * b for a, b in zip(left, right, strict=True))


scores = {name: dot(query, key) / math.sqrt(2) for name, key in keys.items()}
normalizer = sum(math.exp(score) for score in scores.values())
weights = {name: math.exp(score) / normalizer for name, score in scores.items()}

print({name: round(weight, 3) for name, weight in weights.items()})
assert max(weights, key=weights.get) == "get_local_time"
print("最高相关项仍需经过 Registry、Schema 与 Policy；attention != permission")


## 实验 2｜模型先产生 Proposal，工具调用数仍为 0

`ProposalOnlyModel` 只构造一个 `ToolCall` 对象。`FixedClock` 会记录真实 handler 被调用了多少次。请比较两类对象，不要把“模型调用次数”和“工具调用次数”混在一起。


In [ ]:
class FixedClock:
    def __init__(self, value: str = "09:30") -> None:
        self.value = value
        self.calls = 0

    def read(self, zone: str) -> str:
        self.calls += 1
        return self.value


class ProposalOnlyModel:
    def __init__(self) -> None:
        self.calls = 0

    def respond(self, messages: list[dict[str, Any]]) -> Proposal:
        self.calls += 1
        return ToolCall(
            call_id="call_time_1",
            name="get_local_time",
            arguments={"zone": "Asia/Shanghai"},
        )


clock = FixedClock()
proposal_model = ProposalOnlyModel()
proposal = proposal_model.respond([{"role": "user", "content": "上海现在几点？"}])

print(proposal.model_dump(mode="json"))
print({"model_calls": proposal_model.calls, "tool_calls": clock.calls})
assert proposal_model.calls == 1
assert clock.calls == 0


结论不是“框架还没启动”，而是：当前代码只执行了 `model.respond(...)`，尚未执行 `clock.read(...)`。结构化请求仍是数据。

下一步增加三个程序对象：Registry 回答“有没有这个工具”，Schema 回答“参数能否被函数处理”，Policy 回答“当前条件下能不能执行”。


In [ ]:
class GetTimeArgs(BaseModel):
    zone: Literal["Asia/Shanghai"]


class MinutesUntilArgs(BaseModel):
    now: str = Field(pattern=r"^\d{2}:\d{2}$")
    meeting: str = Field(pattern=r"^\d{2}:\d{2}$")


class DeleteCalendarArgs(BaseModel):
    event_id: str = Field(pattern=r"^E-\d+$")


@dataclass(frozen=True)
class ToolSpec:
    name: str
    args_model: type[BaseModel]
    handler: Callable[[BaseModel], Any]
    effect: Literal["read", "write"]


class UnknownToolError(LookupError):
    pass


class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, ToolSpec] = {}

    def register(self, tool: ToolSpec) -> None:
        # TODO 1: 保存工具，并拒绝重复名称。
        raise NotImplementedError

    def resolve(self, name: str) -> ToolSpec:
        # TODO 2: 返回已注册工具；未知名称抛出 UnknownToolError。
        raise NotImplementedError

    def names(self) -> list[str]:
        return sorted(self._tools)


class Principal(BaseModel):
    user_id: str
    can_delete_calendar: bool = False
    confirmed_write: bool = False


class PolicyDecision(BaseModel):
    action: Literal["allow", "block", "wait"]
    rule_id: str


class Policy:
    def decide(self, principal: Principal, tool: ToolSpec) -> PolicyDecision:
        # TODO 3: 只读允许；无写权限 block；未确认 wait；其余 allow。
        raise NotImplementedError


## 实验 2.5｜本地 SQLite：工具怎样取得一条资料

模型提出 `get_calendar_event(event_id="E-7")`，并不等于它已经读到了日历。这里使用 Python 标准库 `sqlite3` 建立内存数据库：表有列，列组成一行事件；Repository 把 SQL 封装成 Python 获取接口；最后才把一行转换为 `CalendarEvent`。

注意数据流的分工：`CalendarRepository` 负责读取，`ToolRegistry` 负责注册名字和参数模型，`Policy` 负责判断主体是否可以使用工具。数据库返回数据与用户获得授权不是同一个判断。


In [ ]:
import sqlite3


@dataclass(frozen=True)
class CalendarEvent:
    event_id: str
    owner_id: str
    title: str
    starts_at: str


class CalendarRepository:
    def __init__(self, connection: sqlite3.Connection) -> None:
        self._connection = connection
        self._connection.row_factory = sqlite3.Row

    def get_event(self, event_id: str, *, owner_id: str) -> CalendarEvent | None:
        # TODO: use a parameterized SELECT with event_id and owner_id.
        # Return None when no row is visible to this owner.
        raise NotImplementedError("实现 SQLite 读取接口")


def build_demo_calendar_repository() -> CalendarRepository:
    connection = sqlite3.connect(":memory:")
    connection.execute(
        '''
        CREATE TABLE calendar_events (
            event_id TEXT PRIMARY KEY,
            owner_id TEXT NOT NULL,
            title TEXT NOT NULL,
            starts_at TEXT NOT NULL
        )
        '''
    )
    connection.executemany(
        "INSERT INTO calendar_events(event_id, owner_id, title, starts_at) VALUES (?, ?, ?, ?)",
        [
            ("E-7", "student", "Agent lesson", "2026-08-13 10:00"),
            ("E-8", "other-user", "Private meeting", "2026-08-13 11:00"),
        ],
    )
    connection.commit()
    return CalendarRepository(connection)


class CalendarReadArgs(BaseModel):
    event_id: str = Field(pattern=r"^E-\d+$")


def make_calendar_reader(
    repository: CalendarRepository,
    *,
    owner_id: str,
) -> Callable[[CalendarReadArgs], dict[str, str] | None]:
    def read(args: CalendarReadArgs) -> dict[str, str] | None:
        event = repository.get_event(args.event_id, owner_id=owner_id)
        if event is None:
            return None
        return {
            "event_id": event.event_id,
            "title": event.title,
            "starts_at": event.starts_at,
        }

    return read


calendar_repository = build_demo_calendar_repository()
calendar_registry = ToolRegistry()
calendar_registry.register(
    ToolSpec(
        name="get_calendar_event",
        args_model=CalendarReadArgs,
        handler=make_calendar_reader(calendar_repository, owner_id="student"),
        effect="read",
    )
)
calendar_tool = calendar_registry.resolve("get_calendar_event")
calendar_policy = Policy().decide(Principal(user_id="student"), calendar_tool)
calendar_value = calendar_tool.handler(
    calendar_tool.args_model.model_validate({"event_id": "E-7"})
)
other_owner_value = calendar_repository.get_event("E-8", owner_id="student")

print({"table": "calendar_events", "columns": ["event_id", "owner_id", "title", "starts_at"]})
print({"registered_db_tools": calendar_registry.names(), "policy": calendar_policy.model_dump()})
print({"student_calendar": calendar_value, "other_owner_row": other_owner_value})


In [ ]:
calendar_side_effects: list[str] = []


def get_local_time(args: GetTimeArgs) -> str:
    return clock.read(args.zone)


def minutes_until(args: MinutesUntilArgs) -> int:
    now = datetime.strptime(args.now, "%H:%M")
    meeting = datetime.strptime(args.meeting, "%H:%M")
    return int((meeting - now).total_seconds() // 60)


def delete_calendar(args: DeleteCalendarArgs) -> str:
    calendar_side_effects.append(args.event_id)
    return f"deleted:{args.event_id}"


registry = ToolRegistry()
registry.register(ToolSpec("get_local_time", GetTimeArgs, get_local_time, "read"))
registry.register(ToolSpec("minutes_until", MinutesUntilArgs, minutes_until, "read"))
registry.register(ToolSpec("delete_calendar", DeleteCalendarArgs, delete_calendar, "write"))

assert registry.names() == ["delete_calendar", "get_local_time", "minutes_until"]
read_decision = Policy().decide(
    Principal(user_id="student"),
    registry.resolve("get_local_time"),
)
write_decision = Policy().decide(
    Principal(user_id="student"),
    registry.resolve("delete_calendar"),
)
assert read_decision.action == "allow"
assert write_decision.action == "block"
print({"registered_tools": registry.names(), "calendar_side_effects": calendar_side_effects})


## 实验 3｜Observation 不是装饰日志，它会改变下一步

`MeetingModel` 不模拟自然语言风格，只根据消息中是否已经出现工具结果来决定下一步：没有时间就请求时间；有时间但没有分钟数就请求计算；两项都有才返回最终答案。

这让我们可以准确比较“回填 Observation”和“删除 Observation”两种状态。


In [ ]:
def latest_observation(
    messages: list[dict[str, Any]], tool_name: str
) -> ToolObservation | None:
    for message in reversed(messages):
        if message.get("role") == "tool" and message.get("name") == tool_name:
            return ToolObservation.model_validate(message["content"])
    return None


class MeetingModel:
    def __init__(self) -> None:
        self.calls = 0

    def respond(self, messages: list[dict[str, Any]]) -> Proposal:
        self.calls += 1
        time_result = latest_observation(messages, "get_local_time")
        minutes_result = latest_observation(messages, "minutes_until")
        if time_result is None:
            return ToolCall(
                call_id=f"time-{self.calls}",
                name="get_local_time",
                arguments={"zone": "Asia/Shanghai"},
            )
        if minutes_result is None:
            return ToolCall(
                call_id=f"minutes-{self.calls}",
                name="minutes_until",
                arguments={"now": time_result.value, "meeting": "10:00"},
            )
        return FinalAnswer(
            answer=f"上海时间为 {time_result.value}，距离会议还有 {minutes_result.value} 分钟。"
        )


messages_without_observation = [{"role": "user", "content": "距离 10:00 还有多久？"}]
first = MeetingModel().respond(messages_without_observation)
second_without_observation = MeetingModel().respond(messages_without_observation)

messages_with_observation = messages_without_observation + [
    {
        "role": "tool",
        "name": "get_local_time",
        "content": ToolObservation(
            call_id="time-1", tool_name="get_local_time", ok=True, value="09:30"
        ).model_dump(mode="json"),
    }
]
second_with_observation = MeetingModel().respond(messages_with_observation)

print({"first": first.name, "without_observation": second_without_observation.name})
print({"with_observation": second_with_observation.name})
assert isinstance(second_without_observation, ToolCall)
assert second_without_observation.name == "get_local_time"
assert isinstance(second_with_observation, ToolCall)
assert second_with_observation.name == "minutes_until"


## 实验 4｜手写有边界的 Runtime

现在把状态、检查、执行和停止接成同一条程序。请特别观察顺序：`resolve → model_validate → policy → limit → handler`。只有最后一步可能调用真实工具。

模型可以返回 final proposal，但 `step_limit`、`tool_call_limit`、`cancelled` 与 `no_progress` 均由程序执行。


In [ ]:
class StopReason(StrEnum):
    COMPLETED = "completed"
    STEP_LIMIT = "step_limit"
    TOOL_CALL_LIMIT = "tool_call_limit"
    CANCELLED = "cancelled"
    NO_PROGRESS = "no_progress"
    UNKNOWN_TOOL = "unknown_tool"
    INVALID_ARGUMENTS = "invalid_arguments"
    POLICY_BLOCKED = "policy_blocked"
    WAITING_FOR_APPROVAL = "waiting_for_approval"
    TOOL_FAILURE = "tool_failure"


class RunLimits(BaseModel):
    max_steps: int = Field(default=6, ge=1)
    max_tool_calls: int = Field(default=4, ge=0)
    max_no_progress_steps: int = Field(default=2, ge=1)


class TraceEvent(BaseModel):
    step: int
    event: str
    data: dict[str, Any] = Field(default_factory=dict)


class RunOutcome(BaseModel):
    status: Literal["passed", "blocked", "failed", "cancelled"]
    stop_reason: StopReason
    answer: str = ""
    steps: int
    tool_calls: int
    messages: list[dict[str, Any]]
    trace: list[TraceEvent]


class CancellationToken:
    def __init__(self) -> None:
        self.cancelled = False

    def cancel(self) -> None:
        self.cancelled = True


def finish(
    reason: StopReason,
    *,
    steps: int,
    tool_calls: int,
    messages: list[dict[str, Any]],
    trace: list[TraceEvent],
    answer: str = "",
) -> RunOutcome:
    # TODO 4: 把 stop reason 映射到 status，并返回完整 RunOutcome。
    raise NotImplementedError


def run_bounded_agent(
    goal: str,
    *,
    model: Model,
    registry: ToolRegistry,
    policy: Policy,
    principal: Principal,
    limits: RunLimits | None = None,
    cancellation: CancellationToken | None = None,
) -> RunOutcome:
    # TODO 5: 按下面的顺序手写循环：
    # cancellation → model → final? → registry → schema → policy
    # → tool-call limit → handler → observation → no-progress → step limit
    # 每个终止分支必须返回明确 StopReason；任何检查失败都不能执行 handler。
    raise NotImplementedError


## 正常路径｜两次工具调用后才完成

先预测完整轨迹：第一轮读时间，第二轮计算，第三轮产生 final answer。随后检查 `steps=3`、`tool_calls=2`，并逐行读取 trace。


In [ ]:
def compact_trace(outcome: RunOutcome) -> list[str]:
    return [
        f"{event.step:02d} | {event.event:<15} | {json.dumps(event.data, ensure_ascii=False)}"
        for event in outcome.trace
    ]


clock.calls = 0
normal = run_bounded_agent(
    "上海现在几点？距离 10:00 的会议还有多少分钟？",
    model=MeetingModel(),
    registry=registry,
    policy=Policy(),
    principal=Principal(user_id="student"),
)
print("\n".join(compact_trace(normal)))
print(normal.model_dump(mode="json", exclude={"messages", "trace"}))

assert normal.status == "passed"
assert normal.stop_reason == StopReason.COMPLETED
assert normal.steps == 3
assert normal.tool_calls == 2
assert "30 分钟" in normal.answer
assert clock.calls == 1


## 拒绝路径｜参数合法仍不能跳过 Policy

这个模型提出一个格式完全正确的删除请求。我们要同时检查三件事：Schema 通过；Policy 返回 block；`calendar_side_effects` 仍为空。只看“拒绝”文字还不够。


In [ ]:
class OneProposalModel:
    def __init__(self, proposal: Proposal) -> None:
        self.proposal = proposal

    def respond(self, messages: list[dict[str, Any]]) -> Proposal:
        return self.proposal


calendar_side_effects.clear()
blocked = run_bounded_agent(
    "删除事件 E-7",
    model=OneProposalModel(
        ToolCall(call_id="delete-1", name="delete_calendar", arguments={"event_id": "E-7"})
    ),
    registry=registry,
    policy=Policy(),
    principal=Principal(user_id="student", can_delete_calendar=False),
)
print("\n".join(compact_trace(blocked)))
print({"stop_reason": blocked.stop_reason, "side_effects": calendar_side_effects})

assert blocked.stop_reason == StopReason.POLICY_BLOCKED
assert blocked.tool_calls == 0
assert calendar_side_effects == []


## 执行前失败｜未知工具与坏参数有不同原因

两条路径都不能运行 handler，但 `stop_reason` 不同。清楚的原因能告诉上层程序该修工具名、改参数，还是请求权限。


In [ ]:
unknown = run_bounded_agent(
    "使用不存在的工具",
    model=OneProposalModel(
        ToolCall(call_id="unknown-1", name="teleport", arguments={"target": "meeting"})
    ),
    registry=registry,
    policy=Policy(),
    principal=Principal(user_id="student"),
)
bad_arguments = run_bounded_agent(
    "读取一个不允许的时区",
    model=OneProposalModel(
        ToolCall(call_id="bad-1", name="get_local_time", arguments={"zone": "Mars/Olympus"})
    ),
    registry=registry,
    policy=Policy(),
    principal=Principal(user_id="student"),
)

print({"unknown": unknown.stop_reason, "bad_arguments": bad_arguments.stop_reason})
assert unknown.stop_reason == StopReason.UNKNOWN_TOOL
assert bad_arguments.stop_reason == StopReason.INVALID_ARGUMENTS
assert unknown.tool_calls == bad_arguments.tool_calls == 0


## 异常停止｜模型永远 continue，也不能决定程序上限

`RepeatingModel` 每轮生成新的 `call_id`，但工具名、参数和结果相同。Runtime 的 no-progress 判断会忽略 `call_id`，关注状态是否真的推进。


In [ ]:
class RepeatingModel:
    def __init__(self) -> None:
        self.calls = 0

    def respond(self, messages: list[dict[str, Any]]) -> Proposal:
        self.calls += 1
        return ToolCall(
            call_id=f"repeat-{self.calls}",
            name="get_local_time",
            arguments={"zone": "Asia/Shanghai"},
        )


step_limited = run_bounded_agent(
    "永远读取时间",
    model=RepeatingModel(),
    registry=registry,
    policy=Policy(),
    principal=Principal(user_id="student"),
    limits=RunLimits(max_steps=2, max_tool_calls=10, max_no_progress_steps=99),
)
no_progress = run_bounded_agent(
    "重复相同动作",
    model=RepeatingModel(),
    registry=registry,
    policy=Policy(),
    principal=Principal(user_id="student"),
    limits=RunLimits(max_steps=10, max_tool_calls=10, max_no_progress_steps=2),
)

print({"step_limit": step_limited.model_dump(exclude={"messages", "trace"})})
print({"no_progress": no_progress.model_dump(exclude={"messages", "trace"})})
assert step_limited.stop_reason == StopReason.STEP_LIMIT
assert step_limited.steps == 2
assert no_progress.stop_reason == StopReason.NO_PROGRESS
assert no_progress.tool_calls == 3


## 工具次数与人工取消｜上限发生在执行前

`max_tool_calls=1` 时，第二个工具请求会被挡在 handler 之前；预先取消时，模型和工具都不应运行。


In [ ]:
tool_limited = run_bounded_agent(
    "完成会议时间任务",
    model=MeetingModel(),
    registry=registry,
    policy=Policy(),
    principal=Principal(user_id="student"),
    limits=RunLimits(max_steps=6, max_tool_calls=1),
)

token = CancellationToken()
token.cancel()
cancelled_model = ProposalOnlyModel()
cancelled = run_bounded_agent(
    "不应开始",
    model=cancelled_model,
    registry=registry,
    policy=Policy(),
    principal=Principal(user_id="student"),
    cancellation=token,
)

print({"tool_limit": tool_limited.stop_reason, "executed": tool_limited.tool_calls})
print({"cancelled": cancelled.stop_reason, "model_calls": cancelled_model.calls})
assert tool_limited.stop_reason == StopReason.TOOL_CALL_LIMIT
assert tool_limited.tool_calls == 1
assert cancelled.stop_reason == StopReason.CANCELLED
assert cancelled_model.calls == 0


## 运行结果可视化｜一次调用中，控制权在哪里转移

下面的时间线直接读取刚才运行得到的 normal、blocked、unknown、tool_limited 和 cancelled。请先按顺序解释每一张卡，再回到 Runtime 找到对应判断。


In [ ]:
from IPython.display import HTML, display

from agent_course.courseware.project_visuals import VisualStep, render_flow

runtime_steps = (
    VisualStep(
        "Model proposal",
        f"steps={normal.steps}",
        "passed",
        lane="Model",
    ),
    VisualStep(
        "Registry + schema",
        f"unknown={unknown.stop_reason.value}",
        "blocked",
        lane="Runtime",
    ),
    VisualStep(
        "Policy",
        blocked.stop_reason.value,
        "blocked",
        lane="Policy",
    ),
    VisualStep(
        "Tool execution",
        f"calls={normal.tool_calls}",
        "passed",
        lane="Application",
    ),
    VisualStep(
        "Observation",
        "tool results returned to messages",
        "passed",
        lane="Runtime",
    ),
    VisualStep(
        "Bounded stop",
        f"{tool_limited.stop_reason.value}; {cancelled.stop_reason.value}",
        "blocked",
        lane="Runtime",
    ),
    VisualStep(
        "Normal terminal",
        normal.stop_reason.value,
        "passed",
        lane="Runtime",
    ),
)
display(
    HTML(
        render_flow(
            title="从 ToolCall 到真实副作用",
            question="哪一张卡才可能让工具调用计数增加？",
            steps=runtime_steps,
            takeaway="模型生成请求；Runtime 检查；应用执行；Observation 回填；上限决定停止。",
        )
    )
)


## 实验 5｜真实 MCP 只替换接线方式

下面使用课程环境中的真实 `MCPServer`，完成工具发现和调用。它仍然调用一个普通 Python 函数；Runtime 的 Policy、次数上限和完成条件没有被 MCP 接管。

本实验在进程内运行，无网络、无凭证、无远程副作用。


In [ ]:
from mcp.server.mcpserver import MCPServer

mcp_clock = FixedClock("09:30")
mcp_server = MCPServer("lesson-01-readonly-tools")


@mcp_server.tool()
def get_local_time_mcp(zone: str) -> str:
    # Return the deterministic course time for one allowed zone.
    if zone != "Asia/Shanghai":
        raise ValueError("unsupported zone")
    return mcp_clock.read(zone)


listed_tools = await mcp_server.list_tools()
mcp_result = await mcp_server.call_tool(
    "get_local_time_mcp", {"zone": "Asia/Shanghai"}
)

print({"tools": [tool.name for tool in listed_tools]})
print({"structured_content": mcp_result.structured_content, "calls": mcp_clock.calls})
assert [tool.name for tool in listed_tools] == ["get_local_time_mcp"]
assert mcp_result.structured_content == {"result": "09:30"}
assert mcp_clock.calls == 1


## Mini project｜公开检查

完成学生 TODO 后，下面九项都应成立：

1. 正常路径执行两个工具后完成；
2. 未知工具在执行前停止；
3. 坏参数在执行前停止；
4. 未授权写操作不产生副作用；
5. step limit 独立于模型；
6. tool-call limit 在 handler 前生效；
7. cancellation 在下一动作前生效；
8. 重复动作触发 no progress；
9. MCP 能发现和调用只读工具，但不替代 Policy。


In [ ]:
public_results = {
    "normal": normal.stop_reason.value,
    "unknown": unknown.stop_reason.value,
    "bad_arguments": bad_arguments.stop_reason.value,
    "blocked": blocked.stop_reason.value,
    "step_limited": step_limited.stop_reason.value,
    "tool_limited": tool_limited.stop_reason.value,
    "cancelled": cancelled.stop_reason.value,
    "no_progress": no_progress.stop_reason.value,
    "mcp_tool": listed_tools[0].name,
}
print(json.dumps(public_results, ensure_ascii=False, indent=2))

assert public_results == {
    "normal": "completed",
    "unknown": "unknown_tool",
    "bad_arguments": "invalid_arguments",
    "blocked": "policy_blocked",
    "step_limited": "step_limit",
    "tool_limited": "tool_call_limit",
    "cancelled": "cancelled",
    "no_progress": "no_progress",
    "mcp_tool": "get_local_time_mcp",
}


## 出口题｜用五个动词重画一次调用

请在自己的笔记中完成：

```text
Model ______ ToolCall
Runtime ______ name / arguments
Registry + Schema + Policy ______ request
Application or MCP Server ______ handler
Runtime ______ Observation and decides continue / stop
```

推荐答案依次是：**生成、解析、检查、执行、回填**。

再回答：如果模型生成字段完全合法的 `delete_all`，副作用前至少还需要检查什么？你的答案应区分工具是否存在、参数/业务规则、权限/用户确认，并指出执行必须在这些检查之后。
